# Your first Puerto Rico agent, in ten minutes

This notebook takes you from nothing to a working competition entry:

1. install the repository (one cell; on Colab nothing else is needed),
2. watch two baseline agents play a game,
3. write an agent of your own, play it against the baselines, and read the numbers,
4. replay a game move by move to see where it went wrong,
5. save the agent as a file and run the official sandbox check on it.

Everything here also works locally: clone the repo, `pip install -e .`, and open this
notebook from the repo root.

[Repository](https://github.com/dae-hany/PuertoRico_AI_Competition) ·
[Game rules](https://github.com/dae-hany/PuertoRico_AI_Competition/blob/main/docs/GAME_RULES.md) ·
[Observation and actions](https://github.com/dae-hany/PuertoRico_AI_Competition/blob/main/docs/OBSERVATION_AND_ACTIONS.md) ·
[Competition rules](https://github.com/dae-hany/PuertoRico_AI_Competition/blob/main/docs/COMPETITION_RULES.md)

In [ ]:
# 1. Setup. On Colab this clones and installs the repository (about a minute).
#    Locally, run the notebook from the repo root and this cell does nothing.
import os, sys

if not os.path.exists("agents"):
    !git clone -q https://github.com/dae-hany/PuertoRico_AI_Competition.git
    %cd PuertoRico_AI_Competition
    !pip -q install -e .
sys.path.insert(0, os.getcwd())

from puerto_rico import make_env, describe_action
from tournament.match import play_game
print("ready")

## 2. A game between two baselines

`play_game` takes a list of agents (2 for the 1-vs-1 track, 3 for the 3p track) and a
seed, plays the whole game under the competition rules, and returns the scores, the
winner, and every decision that was made.

In [ ]:
from agents import ActionValueAgent, TradeBuildingAgent

result = play_game([ActionValueAgent(), TradeBuildingAgent()], seed=1)
print("final VP :", result["scores"], "  winner: seat", result["winners"])
print("decisions:", result["steps"])
print()
for seat, action in result["actions"][:12]:        # the opening, in English
    print(describe_action(seat, action))

## 3. Write your own agent

An agent is a class with one method. `act` receives the observation (a float vector:
220 numbers in the 2p track, 293 in 3p) and the action mask (200 entries, `1` = legal),
and returns the index of one legal action.

The agent below reads two things from the observation, its own money and whether it is
its turn to pick a role, and otherwise follows a fixed preference order over action
groups (see the action table in the docs). It is deliberately simple; make it yours.

In [ ]:
import numpy as np
from agents.base import Agent
from puerto_rico.observation import GLOBAL_DIM, PER_PLAYER_DIM   # 74, 73

class MyAgent(Agent):
    name = "MyFirstAgent"           # shown on the leaderboard: change it

    def act(self, observation, action_mask):
        legal = np.where(action_mask > 0.5)[0]
        me = int(observation[36])                       # which seat I am
        my_block = GLOBAL_DIM + PER_PLAYER_DIM * me
        doubloons = observation[my_block + 23]

        # Role pick (actions 0-7): rich -> Builder, otherwise Craftsman, then Captain
        roles = [a for a in legal if a <= 7]
        if roles:
            for wanted in ([2, 3, 5] if doubloons >= 4 else [3, 5, 2]):
                if wanted in roles:
                    return int(wanted)
            return int(roles[0])

        # Otherwise: ship goods (44-63), then build (16-38), then anything but pass
        for lo, hi in ((44, 63), (16, 38)):
            group = [a for a in legal if lo <= a <= hi]
            if group:
                return int(group[0])
        non_pass = [a for a in legal if a != 15]
        return int(non_pass[0]) if non_pass else 15

# one game to check it runs
r = play_game([MyAgent(), ActionValueAgent()], seed=0)
print("VP:", r["scores"], " winner: seat", r["winners"], " illegal moves:", r["illegal"])

## 4. How good is it? Play a small match

Seats are rotated so that both players get each seat equally (turn order matters in
Puerto Rico). A win rate against `ActionValue` and `TradeBuilding` is the first
milestone; the strongest bundled 2p agent is `SearchAgent`.

Locally, the same thing (with per-move timing) is one command:
`python tools/play.py --agent my_agent.py:MyAgent --vs Search --games 20`

In [ ]:
def match(make_mine, make_opp, games=10, seed=100):
    wins = 0.0
    margins = []
    for g in range(games):
        my_seat = g % 2
        agents = [make_mine(), make_opp()] if my_seat == 0 else [make_opp(), make_mine()]
        r = play_game(agents, seed=seed + g)
        if my_seat in r["winners"]:
            wins += 1.0 / len(r["winners"])
        margins.append(r["scores"][my_seat] - r["scores"][1 - my_seat])
    return 100 * wins / games, float(np.mean(margins))

for name, opp in [("ActionValue", ActionValueAgent), ("TradeBuilding", TradeBuildingAgent)]:
    winpct, margin = match(MyAgent, opp)
    print(f"vs {name:<14} win {winpct:3.0f}%   mean VP margin {margin:+.1f}")

## 5. Replay a lost game

A game is fully described by its seed and its decisions, so it can be stored as a
small JSON record and replayed exactly. Replaying lets you stop at any decision and
look at the position, and the observation, your agent saw.

In [ ]:
from puerto_rico.records import record_from_result, replay_record
from puerto_rico import flatten_observation

r = play_game([MyAgent(), TradeBuildingAgent()], seed=7)
record = record_from_result(r, 2, player_labels=["MyFirstAgent", "TradeBuilding"])
print("VP:", record["scores"], " winner:", record["winners"], " decisions:", len(record["actions"]))

# replay the first 40 decisions and inspect the position the next player sees
env = replay_record(record, stop_after=40)
seat = env.agent_name_mapping[env.agent_selection]
raw = env.observe(env.agent_selection)
obs = flatten_observation(raw["observation"])
legal = np.where(np.asarray(raw["action_mask"]) > 0.5)[0]
print(f"\nafter 40 decisions: seat {seat} to move, {int(obs[GLOBAL_DIM + PER_PLAYER_DIM * seat + 23])} doubloons")
for a in legal[:8]:
    print("  ", describe_action(seat, int(a), env))

## 6. Save it as a file and run the official check

A submission is one `.py` file containing your class. The check below loads the file,
scans its imports, and plays games with your agent **in its own process under the real
1-second deadline**, exactly as the tournament will. It ends with `Result: READY` when
nothing failed.

In [ ]:
%%writefile my_agent.py
import numpy as np
from agents.base import Agent
from puerto_rico.observation import GLOBAL_DIM, PER_PLAYER_DIM

class MyAgent(Agent):
    name = "MyFirstAgent"

    def act(self, observation, action_mask):
        legal = np.where(action_mask > 0.5)[0]
        me = int(observation[36])
        doubloons = observation[GLOBAL_DIM + PER_PLAYER_DIM * me + 23]
        roles = [a for a in legal if a <= 7]
        if roles:
            for wanted in ([2, 3, 5] if doubloons >= 4 else [3, 5, 2]):
                if wanted in roles:
                    return int(wanted)
            return int(roles[0])
        for lo, hi in ((44, 63), (16, 38)):
            group = [a for a in legal if lo <= a <= hi]
            if group:
                return int(group[0])
        non_pass = [a for a in legal if a != 15]
        return int(non_pass[0]) if non_pass else 15

In [ ]:
!python tools/validate_submission.py my_agent.py:MyAgent --games 2

In [ ]:
!python tools/play.py --agent my_agent.py:MyAgent --vs ActionValue,TradeBuilding,SearchLite --games 4

## Where to go from here

- **Plan ahead.** Override `on_game_start(self, forward_model)` to keep the forward
  model, and call `forward_model.clone()` inside `act` to simulate moves. `agents/mcts_agent.py`
  and `agents/search_agent.py` are readable examples; `docs/SEARCH_BASELINE_2P.md` lists
  concrete ways to make the search stronger.
- **Learn.** The environment is a PettingZoo AEC env (`puerto_rico/env.py`); `training/`
  has a PPO self-play trainer. Plain RL still loses to search here, which is the open problem.
- **Play it yourself.** Locally, `python webui/server.py` opens a browser UI where you can
  play against your agent, or watch it play, with every move timed as in the tournament.
- **Submit.** One file per track; see the *The competition* section of the README for
  the dates and the channel, and `docs/SUBMISSION_GUIDE.md` for the checklist.